In [1]:
!pip install -q -U transformers accelerate bitsandbytes fastapi uvicorn pyngrok nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-adk 1.21.0 requires fastapi<0.124.0,>=0.115.0, but you have fastapi 0.132.0 which is incompatible.


In [ ]:
import nest_asyncio
import uvicorn
import asyncio
from pyngrok import ngrok
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import pipeline, BitsAndBytesConfig
import torch

# 1. Setup Ngrok
NGROK_TOKEN = "3A6H7BkDpbzJnEAFNR2Q03KZjbU_5h7ec7EhEaYKNZ69kAhPU" 
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Dual-GPU Quantization Config
# Using float16 compute dtype is best for T4 GPUs
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print("⏳ Loading model across BOTH T4 GPUs...")
model_id = "ayushkumar802/my-finetuned-llm"

# device_map="auto" is the key here—it detects both GPUs automatically
pipe = pipeline(
    "text-generation", 
    model=model_id, 
    model_kwargs={
        "device_map": "auto", 
        "quantization_config": quantization_config,
        "torch_dtype": torch.float16
    }
)

# 3. Create FastAPI App
app = FastAPI()

class Query(BaseModel):
    text: str

@app.post("/generate")
async def generate(query: Query):
    # The pipeline handles the multi-GPU inference logic for you
    output = pipe(query.text, max_new_tokens=200) 
    return {"response": output[0]['generated_text']}

# 4. Start Tunnel
try:
    connection = ngrok.connect(8000)
    public_url = connection.public_url
except:
    public_url = ngrok.get_tunnels()[0].public_url

# 5. Run Server
nest_asyncio.apply()

if __name__ == "__main__":
    print(f"\n🚀 DUAL-GPU SERVER IS LIVE!")
    print(f"!!! USE THIS URL IN STREAMLIT: {public_url}/generate !!!\n")
    
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
    server = uvicorn.Server(config)
    
    # In Kaggle, we can use await server.serve() just like Colab
    await server.serve()

⏳ Loading model across BOTH T4 GPUs...


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:250: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


🚀 DUAL-GPU SERVER IS LIVE!
!!! USE THIS URL IN STREAMLIT: https://peckier-santo-rapturously.ngrok-free.dev/generate !!!



INFO:     Started server process [128]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


...

INFO:     2409:40c4:1a8:4cef:a475:4fe3:3e25:e661:0 - "POST /generate HTTP/1.1" 200 OK


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     2409:40c4:1a8:4cef:a475:4fe3:3e25:e661:0 - "POST /generate HTTP/1.1" 200 OK


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     2409:40c4:1a8:4cef:a475:4fe3:3e25:e661:0 - "POST /generate HTTP/1.1" 200 OK


In [3]:
from pyngrok import ngrok

# 1. Kill any existing ngrok processes
ngrok.kill()

# 2. Reset the tunnels list
tunnels = ngrok.get_tunnels()
for t in tunnels:
    ngrok.disconnect(t.public_url)

print("✅ Ngrok cleaned up. You can now run the server cell.")

✅ Ngrok cleaned up. You can now run the server cell.
